# Create the dataset for computation

In [1]:
import datasets
import pandas as pd
import numpy as np

In [2]:
txt_only = False
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{include_embeddings}_shared_weights"

In [3]:
window = 28
mod = 4
max_periods = 54

data_files = {
    "train": ["train-00000-of-00002.parquet", "train-00001-of-00002.parquet"],
    "validation": [
        "validation-00000-of-00002.parquet",
        "validation-00001-of-00002.parquet",
    ],
}


ds_dict = datasets.load_dataset(
    "parquet",
    data_dir="../../../main/data/amzn_toys_monthly_diffs_ffill_fixed_splits_buybox",
    data_files=data_files,
)

ds_val = ds_dict['validation']
ds_train = ds_dict['train']

columns = ['ASIN',
           'SALES_RANK',
           'PRICE',
           'BUYBOX_PRICE',
           'text',
           'date',
           'window',
           'REVIEW_COUNT',
           'RATING',
           'New Offer Count: Current',
           'Count of retrieved live offers: New, FBA',
           'Count of retrieved live offers: New, FBM',
           'Lightning Deals: Upcoming Deal',
           'Buy Box: Is FBA',
           'subcat_aggregated']

df_val = ds_val.select_columns(columns).to_pandas()
df_val = df_val[df_val["window"] == window].dropna()
df_val["date_t"] = df_val["date"].astype("category").cat.codes
df_val = df_val[df_val["date_t"] <= max_periods]
df_val = df_val[df_val["date_t"] % mod == 0]

df_train = ds_train.select_columns(columns).to_pandas()
df_train = df_train[df_train["window"] == window].dropna()
df_train["date_t"] = df_train["date"].astype("category").cat.codes
df_train = df_train[df_train["date_t"] <= max_periods]
df_train = df_train[df_train["date_t"] % mod == 0]


df_train_val = pd.concat([df_train, df_val], axis=0).reset_index()

# Dataset contains data with different averaged values (by window size).
# Predictions / Embeddings only for one specific window size

num_val = len(df_val)
num_train = len(df_train)

print(f"Number of train samples: {num_train}")
print(f"Number of val samples: {num_val}")

# Create dummy variables for the subcategories
dummy_subcat = pd.get_dummies(df_train_val["subcat_aggregated"]) * 1
dummy_subcat_names = list(dummy_subcat.columns)
dummy_time = pd.get_dummies(df_train_val["date"]) * 1
dummy_time_names = list(dummy_time.columns)

df_train_val = pd.concat([df_train_val, dummy_subcat, dummy_time], axis=1)

df_full = pd.concat([df_train_val, dummy_subcat], axis=1)
df_full.set_index(["ASIN", "date"], inplace=True)

print(f"df_full shape: {df_full.shape}")
print(df_full.columns)

Number of train samples: 41945
Number of val samples: 41832
df_full shape: (83777, 51)
Index(['index', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       'Count of retrieved live offers: New, FBM',
       'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA',
       'subcat_aggregated', 'date_t', 'Airplanes', 'Cars & Race Cars',
       'Motor Vehicles', 'Race Tracks', 'Residual', 'Skateboards', 'Tractors',
       'Train Sets', 'Trains & Trams', 'Trucks', 'Vehicle Playsets',
       '2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22',
       '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09',
       '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29', 'Airplanes',
       'Cars & Race Cars', 'Motor Vehicles', 'Race Tracks', 'Residual',
       'Skateboards', 'Tractors', 'Train Sets', 'Trains & Trams', 'Trucks',
       'Vehicle Play

In [4]:
import os
os.chdir("../..")

train_preds = pd.read_csv("data/A3_dimensionality_reduction_via_NN/data/train_pred_model-2025-11-25-embdim=5_lag1_txtimg_mod4-epoch=070-val_combined_loss=0.3976_txtimg_with_last_layer_dim=5_full_emb_step=.csv")
valid_preds = pd.read_csv("data/A3_dimensionality_reduction_via_NN/data/val_pred_model-2025-11-25-embdim=5_lag1_txtimg_mod4-epoch=070-val_combined_loss=0.3976_txtimg_with_last_layer_dim=5_full_emb_step=.csv")

In [5]:
print(f"train_preds shape: {train_preds.shape}")
print(f"valid_preds shape: {valid_preds.shape}")

train_preds shape: (38309, 14)
valid_preds shape: (38203, 14)


In [6]:
train_preds.head()

,index,time,pred_ml_l,pred_ml_m,embedding_feature_0,embedding_feature_1,embedding_feature_2,embedding_feature_3,embedding_feature_4,emb_0,emb_1,emb_2,emb_3,emb_4
0,1947335162,2023-02-27,-12.599700,3.578376,-0.955177,-0.907097,2.764652,-0.999840,-0.918854,-0.955177,-0.907097,2.764652,-0.999840,-0.918854
1,1947335162,2023-03-27,-12.599700,3.578376,-0.955177,-0.907097,2.764652,-0.999840,-0.918854,-0.955177,-0.907097,2.764652,-0.999840,-0.918854
2,1947335162,2023-04-24,-12.592524,3.578664,-0.954928,-0.906611,2.761426,-0.999838,-0.918442,-0.954928,-0.906611,2.761426,-0.999838,-0.918442
3,1947335162,2023-05-22,-12.599520,3.578375,-0.955175,-0.907081,2.764575,-0.999839,-0.918840,-0.955175,-0.907081,2.764575,-0.999839,-0.918840
4,1947335162,2023-06-19,-12.598991,3.578404,-0.955160,-0.907033,2.764333,-0.999839,-0.918824,-0.955160,-0.907033,2.764333,-0.999839,-0.918824


In [7]:
preds = pd.concat([train_preds, valid_preds], axis=0).reset_index(drop=True).rename(columns={"index": "ASIN", "time": "date"}).set_index(["ASIN", "date"])
# rename index to asin
preds.head(-10)

pred_ml_l  pred_ml_m  embedding_feature_0  \
ASIN       date                                                    
1947335162 2023-02-27 -12.599700   3.578376            -0.955177   
           2023-03-27 -12.599700   3.578376            -0.955177   
           2023-04-24 -12.592524   3.578664            -0.954928   
           2023-05-22 -12.599520   3.578375            -0.955175   
           2023-06-19 -12.598991   3.578404            -0.955160   
...                          ...        ...                  ...   
B0CMX99LNN 2024-01-29 -11.456852   3.581485            -0.911516   
B0CMXQP6NS 2024-01-29 -10.072104   3.137301            -0.773646   
B0CNCPNTNJ 2024-01-29  -9.676799   3.340877            -0.807553   
B0CNH8PS9N 2024-01-29 -11.260119   2.591002            -0.705784   
B0CNN9B7XG 2024-01-29 -10.537450   2.768028            -0.703423   

                       embedding_feature_1  embedding_feature_2  \
ASIN       date                                                   
1947335162 2023-02-27            -0.907097             2.764652   
           2023-03-27            -0.907097             2.764652   
           2023-04-24            -0.906611             2.761426   
           2023-05-22            -0.907081             2.764575   
           2023-06-19            -0.907033             2.764333   
...                                    ...                  ...   
B0CMX99LNN 2024-01-29            -0.829280             2.275619   
B0CMXQP6NS 2024-01-29            -0.705052             1.944210   
B0CNCPNTNJ 2024-01-29            -0.692816             1.656983   
B0CNH8PS9N 2024-01-29            -0.731196             2.766400   
B0CNN9B7XG 2024-01-29            -0.704352             2.357816   

                       embedding_feature_3  embedding_feature_4     emb_0  \
ASIN       date                                                             
1947335162 2023-02-27            -0.999840            -0.918854 -0.955177   
           2023-03-27            -0.999840            -0.918854 -0.955177   
           2023-04-24            -0.999838            -0.918442 -0.954928   
           2023-05-22            -0.999839            -0.918840 -0.955175   
           2023-06-19            -0.999839            -0.918824 -0.955160   
...                                    ...                  ...       ...   
B0CMX99LNN 2024-01-29            -0.999135            -0.837176 -0.911516   
B0CMXQP6NS 2024-01-29            -0.993936            -0.640046 -0.773646   
B0CNCPNTNJ 2024-01-29            -0.992898            -0.646514 -0.807553   
B0CNH8PS9N 2024-01-29            -0.996890            -0.630413 -0.705784   
B0CNN9B7XG 2024-01-29            -0.994548            -0.616165 -0.703423   

                          emb_1     emb_2     emb_3     emb_4  
ASIN       date                                                
1947335162 2023-02-27 -0.907097  2.764652 -0.999840 -0.918854  
           2023-03-27 -0.907097  2.764652 -0.999840 -0.918854  
           2023-04-24 -0.906611  2.761426 -0.999838 -0.918442  
           2023-05-22 -0.907081  2.764575 -0.999839 -0.918840  
           2023-06-19 -0.907033  2.764333 -0.999839 -0.918824  
...                         ...       ...       ...       ...  
B0CMX99LNN 2024-01-29 -0.829280  2.275619 -0.999135 -0.837176  
B0CMXQP6NS 2024-01-29 -0.705052  1.944210 -0.993936 -0.640046  
B0CNCPNTNJ 2024-01-29 -0.692816  1.656983 -0.992898 -0.646514  
B0CNH8PS9N 2024-01-29 -0.731196  2.766400 -0.996890 -0.630413  
B0CNN9B7XG 2024-01-29 -0.704352  2.357816 -0.994548 -0.616165  

[76502 rows x 12 columns]

In [8]:
df_full

index  SALES_RANK     PRICE  BUYBOX_PRICE  \
ASIN       date                                                     
1947335162 2023-01-30       1  -12.747395  3.400864      3.555062   
           2023-02-27       9  -12.600586  3.404632      3.555062   
           2023-03-27      17  -12.515369  3.427422      3.633271   
           2023-04-24      25  -12.576837  3.400864      3.582963   
           2023-05-22      33  -12.688608  3.400864      3.587493   
...                       ...         ...       ...           ...   
B0CQWJ3247 2024-01-01  592585   -9.410761  2.078191      2.078191   
           2024-01-29  592593   -9.367674  2.116772      2.116772   
B0CQWJZGSY 2024-01-01  592741   -9.376208  1.885553      1.885553   
           2024-01-29  592749   -9.461319  1.998193      1.998193   
B0CRDLRP38 2024-01-29  592905  -11.850726  3.295466      3.295466   

                                                                    text  \
ASIN       date                                                            
1947335162 2023-01-30  Catalyst Game Labs Battletech: Technical Reado...   
           2023-02-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-03-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-04-24  Catalyst Game Labs Battletech: Technical Reado...   
           2023-05-22  Catalyst Game Labs Battletech: Technical Reado...   
...                                                                  ...   
B0CQWJ3247 2024-01-01  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
           2024-01-29  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
B0CQWJZGSY 2024-01-01  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
           2024-01-29  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
B0CRDLRP38 2024-01-29  Keejoy 2-in-1 Dinosaur Truck Toy with 6 Stylis...   

                       window  REVIEW_COUNT    RATING  \
ASIN       date                                         
1947335162 2023-01-30    28.0     62.000000  4.700000   
           2023-02-27    28.0     62.000000  4.700000   
           2023-03-27    28.0     62.035714  4.678571   
           2023-04-24    28.0     63.607143  4.700000   
           2023-05-22    28.0     64.000000  4.700000   
...                       ...           ...       ...   
B0CQWJ3247 2024-01-01    28.0      3.000000  3.500000   
           2024-01-29    28.0      2.357143  3.982143   
B0CQWJZGSY 2024-01-01    28.0      3.000000  3.500000   
           2024-01-29    28.0      2.357143  3.982143   
B0CRDLRP38 2024-01-29    28.0     11.166667  3.708696   

                       New Offer Count: Current  \
ASIN       date                                   
1947335162 2023-01-30                        10   
           2023-02-27                        10   
           2023-03-27                        10   
           2023-04-24                        10   
           2023-05-22                        10   
...                                         ...   
B0CQWJ3247 2024-01-01                         1   
           2024-01-29                         1   
B0CQWJZGSY 2024-01-01                         1   
           2024-01-29                         1   
B0CRDLRP38 2024-01-29                         1   

                       Count of retrieved live offers: New, FBA  ...  \
ASIN       date                                                  ...   
1947335162 2023-01-30                                         2  ...   
           2023-02-27                                         2  ...   
           2023-03-27                                         2  ...   
           2023-04-24                                         2  ...   
           2023-05-22                                         2  ...   
...                                                         ...  ...   
B0CQWJ3247 2024-01-01                                         0  ...   
           2024-01-29                                         0  ...   
B0CQWJZGSY 

In [9]:
df_full = df_full.join(preds)
df_full.head(-10)

index  SALES_RANK     PRICE  BUYBOX_PRICE  \
ASIN       date                                                     
1947335162 2023-01-30       1  -12.747395  3.400864      3.555062   
           2023-02-27       9  -12.600586  3.404632      3.555062   
           2023-03-27      17  -12.515369  3.427422      3.633271   
           2023-04-24      25  -12.576837  3.400864      3.582963   
           2023-05-22      33  -12.688608  3.400864      3.587493   
...                       ...         ...       ...           ...   
B0CPDVS4T1 2024-01-29  591501  -10.785137  3.239924      3.239924   
B0CPSR7XM1 2024-01-01  591649  -11.077371  3.331847      3.327371   
           2024-01-29  591657  -11.601493  3.036690      3.036690   
B0CQ1SLW1P 2024-01-01  591805  -11.610195  2.396986      2.563410   
           2024-01-29  591813  -12.030145  2.396986      2.563410   

                                                                    text  \
ASIN       date                                                            
1947335162 2023-01-30  Catalyst Game Labs Battletech: Technical Reado...   
           2023-02-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-03-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-04-24  Catalyst Game Labs Battletech: Technical Reado...   
           2023-05-22  Catalyst Game Labs Battletech: Technical Reado...   
...                                                                  ...   
B0CPDVS4T1 2024-01-29  FUUY RC Excavator Toy for Boys Toys Age 4-6, R...   
B0CPSR7XM1 2024-01-01  Wbzle Spaceship Shuttle Toy with Astronaut Fig...   
           2024-01-29  Wbzle Spaceship Shuttle Toy with Astronaut Fig...   
B0CQ1SLW1P 2024-01-01  FLAT 2 FAST Red 72 Card Racer | Load, Launch, ...   
           2024-01-29  FLAT 2 FAST Red 72 Card Racer | Load, Launch, ...   

                       window  REVIEW_COUNT    RATING  \
ASIN       date                                         
1947335162 2023-01-30    28.0     62.000000  4.700000   
           2023-02-27    28.0     62.000000  4.700000   
           2023-03-27    28.0     62.035714  4.678571   
           2023-04-24    28.0     63.607143  4.700000   
           2023-05-22    28.0     64.000000  4.700000   
...                       ...           ...       ...   
B0CPDVS4T1 2024-01-29    28.0      5.814815  4.933333   
B0CPSR7XM1 2024-01-01    28.0     27.947368  4.505263   
           2024-01-29    28.0     29.714286  4.407143   
B0CQ1SLW1P 2024-01-01    28.0      1.000000  4.000000   
           2024-01-29    28.0      1.000000  4.000000   

                       New Offer Count: Current  \
ASIN       date                                   
1947335162 2023-01-30                        10   
           2023-02-27                        10   
           2023-03-27                        10   
           2023-04-24                        10   
           2023-05-22                        10   
...                                         ...   
B0CPDVS4T1 2024-01-29                         1   
B0CPSR7XM1 2024-01-01                         2   
           2024-01-29                         2   
B0CQ1SLW1P 2024-01-01                         1   
           2024-01-29                         1   

                       Count of retrieved live offers: New, FBA  ...  \
ASIN       date                                                  ...   
1947335162 2023-01-30                                         2  ...   
           2023-02-27                                         2  ...   
           2023-03-27                                         2  ...   
           2023-04-24                                         2  ...   
           2023-05-22                                         2  ...   
...                                                         ...  ...   
B0CPDVS4T1 2024-01-29                                         1  ...   
B0CPSR7XM1 2024-01-01                                         0  ...   
           

In [10]:
df_full_val = df_full.iloc[num_train:, :]
df_full_train = df_full.iloc[:num_train, :]

print(f"df_full shape: {df_full.shape}")
print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")
print(df_full.columns)

df_full shape: (83777, 63)
df_full_train shape: (41945, 63)
df_full_val shape: (41832, 63)
Index(['index', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       'Count of retrieved live offers: New, FBM',
       'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA',
       'subcat_aggregated', 'date_t', 'Airplanes', 'Cars & Race Cars',
       'Motor Vehicles', 'Race Tracks', 'Residual', 'Skateboards', 'Tractors',
       'Train Sets', 'Trains & Trams', 'Trucks', 'Vehicle Playsets',
       '2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22',
       '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09',
       '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29', 'Airplanes',
       'Cars & Race Cars', 'Motor Vehicles', 'Race Tracks', 'Residual',
       'Skateboards', 'Tractors', 'Train Sets', 'Trains & Trams', 'Trucks',
       'Vehicle 

In [11]:
print(df_full_val.isna().sum().sort_values(ascending=False).head(20))

emb_2                  3629
emb_3                  3629
emb_4                  3629
embedding_feature_1    3629
embedding_feature_0    3629
pred_ml_m              3629
pred_ml_l              3629
emb_1                  3629
embedding_feature_4    3629
embedding_feature_3    3629
embedding_feature_2    3629
emb_0                  3629
index                     0
SALES_RANK                0
PRICE                     0
Airplanes                 0
Cars & Race Cars          0
Motor Vehicles            0
Race Tracks               0
BUYBOX_PRICE              0
dtype: int64


In [12]:
print(df_full_train.isna().sum().sort_values(ascending=False).head(20))

emb_2                  3636
emb_3                  3636
emb_4                  3636
embedding_feature_1    3636
embedding_feature_0    3636
pred_ml_m              3636
pred_ml_l              3636
emb_1                  3636
embedding_feature_4    3636
embedding_feature_3    3636
embedding_feature_2    3636
emb_0                  3636
index                     0
SALES_RANK                0
PRICE                     0
Airplanes                 0
Cars & Race Cars          0
Motor Vehicles            0
Race Tracks               0
BUYBOX_PRICE              0
dtype: int64


In [13]:
df_full_val = df_full_val.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
for i in range(1, 3):
    df_full_val[f"Q_t-{i}"] = df_full_val.groupby("ASIN")["Q_t"].shift(i)
    df_full_val[f"P_bb_t-{i}"] = df_full_val.groupby("ASIN")["P_bb_t"].shift(i)
    df_full_val[f"REVIEW_COUNT_t-{i}"] = df_full_val["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_val[f"RATING_t-{i}"] = df_full_val["RATING"].groupby("ASIN").shift(i)

df_full_val["Delta_Q_t"] = df_full_val["Q_t"] - df_full_val["Q_t-1"]
df_full_val["Delta_P_bb_t"] = df_full_val["P_bb_t"] - df_full_val["P_bb_t-1"]


df_full_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}, inplace=True)
for i in range(1, 3):
    df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
    df_full_train[f"P_bb_t-{i}"] = df_full_train["P_bb_t"].groupby("ASIN").shift(i)
    df_full_train[f"REVIEW_COUNT_t-{i}"] = df_full_train["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_train[f"RATING_t-{i}"] = df_full_train["RATING"].groupby("ASIN").shift(i)
    
df_full_train["Delta_Q_t"] = df_full_train["Q_t"] - df_full_train["Q_t-1"]
df_full_train["Delta_P_bb_t"] = df_full_train["P_bb_t"] - df_full_train["P_bb_t-1"]

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")

print(f"df_full_train columns: {df_full_train.columns}")
print(f"df_full_val columns: {df_full_val.columns}")

df_full_train shape: (41945, 73)
df_full_val shape: (41832, 73)
df_full_train columns: Index(['index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       'Count of retrieved live offers: New, FBM',
       'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA',
       'subcat_aggregated', 'date_t', 'Airplanes', 'Cars & Race Cars',
       'Motor Vehicles', 'Race Tracks', 'Residual', 'Skateboards', 'Tractors',
       'Train Sets', 'Trains & Trams', 'Trucks', 'Vehicle Playsets',
       '2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22',
       '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09',
       '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29', 'Airplanes',
       'Cars & Race Cars', 'Motor Vehicles', 'Race Tracks', 'Residual',
       'Skateboards', 'Tractors', 'Train Sets', 'Trains & Trams', 'Trucks',
       'Vehicle Playsets', 'pred_

C:\Users\Work\AppData\Local\Temp\ipykernel_25908\1382120059.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}, inplace=True)
C:\Users\Work\AppData\Local\Temp\ipykernel_25908\1382120059.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
C:\Users\Work\AppData\Local\Temp\ipykernel_25908\1382120059.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

In [15]:
df_full_val.reset_index()["date"].dropna().unique()

array(['2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24',
       '2023-05-22', '2023-06-19', '2023-07-17', '2023-08-14',
       '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04',
       '2024-01-01', '2024-01-29'], dtype=object)

write to csv 

In [16]:
drop_cols=[
    'Q_t-2',
    'P_bb_t-2',
    'REVIEW_COUNT_t-2',
    'RATING_t-2',
    'pred_ml_l_lag_1',
    'pred_ml_m_lag_1',
]

keep_cols = [c for c in df_full_val.columns if c not in drop_cols]


In [17]:
print(df_full_val[keep_cols].isna().sum().sort_values(ascending=False).head(20))
print("\n\n")
print(df_full_train[keep_cols].isna().sum().sort_values(ascending=False).head(20))

REVIEW_COUNT_t-1       3775
P_bb_t-1               3775
Q_t-1                  3775
RATING_t-1             3775
Delta_P_bb_t           3775
Delta_Q_t              3775
emb_2                  3629
emb_1                  3629
emb_0                  3629
pred_ml_m              3629
pred_ml_l              3629
embedding_feature_1    3629
embedding_feature_0    3629
embedding_feature_2    3629
embedding_feature_3    3629
embedding_feature_4    3629
emb_4                  3629
emb_3                  3629
index                     0
P_bb_t                    0
dtype: int64



REVIEW_COUNT_t-1       3774
P_bb_t-1               3774
Q_t-1                  3774
RATING_t-1             3774
Delta_P_bb_t           3774
Delta_Q_t              3774
emb_2                  3636
emb_1                  3636
emb_0                  3636
pred_ml_m              3636
pred_ml_l              3636
embedding_feature_1    3636
embedding_feature_0    3636
embedding_feature_2    3636
embedding_feature_3    3636
embe

In [18]:
list(df_full_val.columns)

['index',
 'Q_t',
 'PRICE',
 'P_bb_t',
 'text',
 'window',
 'REVIEW_COUNT',
 'RATING',
 'New Offer Count: Current',
 'Count of retrieved live offers: New, FBA',
 'Count of retrieved live offers: New, FBM',
 'Lightning Deals: Upcoming Deal',
 'Buy Box: Is FBA',
 'subcat_aggregated',
 'date_t',
 'Airplanes',
 'Cars & Race Cars',
 'Motor Vehicles',
 'Race Tracks',
 'Residual',
 'Skateboards',
 'Tractors',
 'Train Sets',
 'Trains & Trams',
 'Trucks',
 'Vehicle Playsets',
 '2023-01-30',
 '2023-02-27',
 '2023-03-27',
 '2023-04-24',
 '2023-05-22',
 '2023-06-19',
 '2023-07-17',
 '2023-08-14',
 '2023-09-11',
 '2023-10-09',
 '2023-11-06',
 '2023-12-04',
 '2024-01-01',
 '2024-01-29',
 'Airplanes',
 'Cars & Race Cars',
 'Motor Vehicles',
 'Race Tracks',
 'Residual',
 'Skateboards',
 'Tractors',
 'Train Sets',
 'Trains & Trams',
 'Trucks',
 'Vehicle Playsets',
 'pred_ml_l',
 'pred_ml_m',
 'embedding_feature_0',
 'embedding_feature_1',
 'embedding_feature_2',
 'embedding_feature_3',
 'embedding_feat

In [21]:
main_keys_val = pd.read_csv("../main/code/main_val_keys.csv")
main_keys_train = pd.read_csv("../main/code/main_train_keys.csv")

In [22]:
# filter for keys in may keys

df_full_val = df_full_val.reset_index().merge(main_keys_val, on=["ASIN", "date"], how="inner").set_index(["ASIN", "date"])
df_full_train = df_full_train.reset_index().merge(main_keys_train, on=["ASIN", "date"], how="inner").set_index(["ASIN", "date"])

In [23]:
df_full_val.shape, df_full_train.shape

((38041, 73), (38146, 73))

In [26]:
zip_train = dict(method="zip", archive_name=f"data/A3_dimensionality_reduction_via_NN/{dataframe_name}_train.csv")
df_full_train.to_csv(f"data/A3_dimensionality_reduction_via_NN/{dataframe_name}_train.zip", compression=zip_train)
zip_val = dict(method="zip", archive_name=f"data/A3_dimensionality_reduction_via_NN/{dataframe_name}_val.csv")
df_full_val.to_csv(f"data/A3_dimensionality_reduction_via_NN/{dataframe_name}_val.zip", compression=zip_val)